# dim_google_review_scd1 (SCD 1)

One row per Google review (Place Details, accumulated). `restaurant_id` = current `dim_restaurant` row of the place (`'unknown'` if none). `review_key = xxhash64(review_id)`.

Task `dim_google_review_scd1` of the job "2. gold (poc tables)"; widgets `load_type` (incremental / full), `from_date`, `to_date` (Vietnam days, empty = yesterday .. tomorrow). The live dim is unchanged.

In [ ]:
%run ./_scd

# Set up environment

In [ ]:
for layer in ("silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])
dbutils.widgets.text("from_date", "")   # yyyy-MM-dd, Vietnam day, inclusive; empty = yesterday
dbutils.widgets.text("to_date", "")     # exclusive; empty = tomorrow

silver = dbutils.widgets.get("silver")
gold = dbutils.widgets.get("gold")
load_type = dbutils.widgets.get("load_type")
from_date, to_date = get_incremental_dates(dbutils.widgets.get("from_date"), dbutils.widgets.get("to_date"))

# First load of the table = full (poc update_load_type: full once, then incremental)
if not spark.catalog.tableExists(f"{gold}.dim_google_review_scd1"):
    load_type = "full"
print(f"load_type = {load_type}, window = [{from_date}, {to_date})")

# Read Source Table

Skipped until `stg_restaurant_place_reviews` exists.

In [ ]:
path = f"{gold}.dim_google_review_scd1"
reviews_table = f"{silver}.stg_restaurant_place_reviews"
if not spark.catalog.tableExists(reviews_table):
    dbutils.notebook.exit(f"{reviews_table} does not exist yet: {path} not built")

reviews = spark.table(reviews_table).filter("review_id IS NOT NULL")
restaurants = (spark.table(f"{gold}.dim_restaurant")
               .filter("is_current AND restaurant_id <> 'unknown' AND place_id IS NOT NULL")
               .groupBy("place_id").agg(F.min("restaurant_id").alias("restaurant_id")))

## Incremental: reviews read in the window **+** reviews still without a restaurant

In [ ]:
if load_type == "incremental":
    seen_day = F.to_date(F.from_utc_timestamp("last_seen_at", TZ))
    changed = reviews.filter((seen_day >= from_date) & (seen_day < to_date)).select("review_id")
    if spark.catalog.tableExists(path):
        unresolved = spark.table(path).filter("review_key <> -1 AND restaurant_id = 'unknown'").select("review_id")
        changed = changed.union(unresolved)
    reviews = reviews.join(changed.distinct(), "review_id", "left_semi")

# Transformation

In [ ]:
result_df = (
    reviews.alias("review")
    .join(restaurants.alias("restaurant"), on=F.col("review.place_id") == F.col("restaurant.place_id"), how="left")
    .select(
        F.xxhash64("review.review_id").alias("review_key"),
        F.col("review.review_id").cast("string").alias("review_id"),
        F.coalesce(F.col("restaurant.restaurant_id"), F.lit("unknown")).alias("restaurant_id"),
        F.col("review.place_id").cast("string").alias("place_id"),
        F.coalesce(F.date_format(F.from_utc_timestamp("review.publish_time", TZ), "yyyyMMdd").cast("int"),
                   F.lit(-1)).alias("publish_date_key"),
        F.col("review.rating").cast("tinyint").alias("rating"),
        F.col("review.review_rank").cast("int").alias("review_rank"),
        F.col("review.review_text").cast("string").alias("review_text"),
        F.col("review.review_language").cast("string").alias("review_language"),
        F.col("review.original_text").cast("string").alias("original_text"),
        F.col("review.original_language").cast("string").alias("original_language"),
        F.col("review.author_name").cast("string").alias("author_name"),
        F.col("review.publish_time").cast("timestamp").alias("publish_time"),
        F.col("review.relative_time").cast("string").alias("relative_time"),
        F.col("review.google_maps_uri").cast("string").alias("google_maps_uri"),
        F.col("review.first_seen_at").cast("timestamp").alias("created_at"),
        F.col("review.last_seen_at").cast("timestamp").alias("modified_at"),
    )
)

## Add metadata

In [ ]:
insert_date = datetime.now()
etl_by = "Databricks"
result_df = result_df.withColumn("etl_date", F.lit(insert_date)).withColumn("etl_by", F.lit(etl_by))

## Add checksums

In [ ]:
primary_key = ["review_id"]
result_df = add_checksums(result_df, primary_key, exclude=["review_key", "created_at", "modified_at"])

## Create table

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {path} (
        review_key BIGINT, review_id STRING, restaurant_id STRING, place_id STRING, publish_date_key INT, rating TINYINT,
        review_rank INT, review_text STRING, review_language STRING, original_text STRING, original_language STRING,
        author_name STRING, publish_time TIMESTAMP, relative_time STRING, google_maps_uri STRING,
        created_at TIMESTAMP, modified_at TIMESTAMP, etl_date TIMESTAMP, etl_by STRING,
        primary_checksum STRING, record_checksum STRING
    ) USING DELTA
    COMMENT 'Google review dimension, SCD 1 (poc checksum method). review_key = xxhash64(review_id), -1 = Unknown.'
""")
result_df = result_df.select(*spark.table(path).columns)

# Write to Table

**full**: overwrite (poc `full_refresh`). **incremental**: MERGE on `primary_checksum`; update only when `record_checksum` changed and the source row is not older (`s.modified_at >= t.modified_at`, so a backfill of an old day never overwrites newer values); insert new keys.

In [ ]:
if load_type == "full":
    full_refresh(result_df, path)
else:
    result_df.createOrReplaceTempView("source")
    spark.sql(f"""
        MERGE INTO {path} AS t
        USING source AS s
        ON t.primary_checksum = s.primary_checksum
        WHEN MATCHED AND t.record_checksum <> s.record_checksum
             AND (t.modified_at IS NULL OR s.modified_at >= t.modified_at) THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

## Unknown row

Key −1, natural key NULL: facts with no match point to it. Inserted only when missing (no empty commit).

In [ ]:
if spark.table(path).where("review_key = -1").isEmpty():
    spark.sql(f"INSERT INTO {path} (review_key, restaurant_id, publish_date_key, review_text, etl_date, etl_by) VALUES (-1, 'unknown', -1, 'Unknown', current_timestamp(), 'Databricks')")

## Check

In [ ]:
dup = spark.sql(f"SELECT COUNT(*) - COUNT(DISTINCT review_id) FROM {path} WHERE review_id IS NOT NULL").first()[0]
assert dup == 0, f"{path}: {dup} duplicate review_id"
last = spark.sql(f"DESCRIBE HISTORY {path} LIMIT 2").where("operation <> 'WRITE'").select("operation", "operationMetrics").first()
print(f"✅ {path}: {spark.table(path).count()} rows", "·", last["operation"], dict(last["operationMetrics"] or {}) if last else "")